# ClawHub Skill Ecosystem Analysis

**Corpus**: 13,729 skills from ClawHub registry (Feb 2026 snapshot)  
**Question**: What do agent capabilities look like in practice, and why do they fail?

This notebook reproduces the key findings from the effectorHQ analysis.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_DIR = Path('../data')

## 1. Type Frequency Distribution

What types do skills actually use? We inferred input/output/context types from frontmatter + NLP analysis of SKILL.md bodies.

In [ ]:
with open(DATA_DIR / 'type-frequency.json') as f:
    freq = json.load(f)

print(f"Corpus size: {freq['metadata']['corpus_size']:,} skills")
print(f"Snapshot: {freq['metadata']['snapshot_date']}")
print(f"Overall failure rate: {freq['metadata']['overall_failure_rate']:.0%}")
print(f"Skills with no typed interface: {freq['metadata']['skills_with_no_typed_interface']:.0%}")
print(f"Skills with permission over-declaration: {freq['metadata']['skills_with_permission_overdeclaration']:.0%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (category, data) in zip(axes, [('Input', freq['input']), ('Output', freq['output']), ('Context', freq['context'])]):
    types = list(data.keys())
    values = [v * 100 for v in data.values()]
    colors = sns.color_palette('Reds_r', len(types))
    ax.barh(types[::-1], values[::-1], color=colors)
    ax.set_xlabel('Frequency (%)')
    ax.set_title(f'{category} Type Distribution', fontweight='bold')
    for i, (t, v) in enumerate(zip(types[::-1], values[::-1])):
        ax.text(v + 0.5, i, f'{v:.0f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../figures/type-distributions-combined.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Failure Rate by Cluster

We clustered skills by inferred interface pattern (k=12, silhouette=0.61), then measured success rates in sandboxed execution.

In [ ]:
clusters = pd.read_csv(DATA_DIR / 'failure-rates-by-cluster.csv')
clusters['failure_rate'] = 1 - clusters['success_rate']

print(f"Total sampled: {clusters['n_sampled'].sum():,}")
print(f"Total untestable: {clusters['n_untestable'].sum():,}")
print(f"Effective sample: {clusters['n_sampled'].sum() - clusters['n_untestable'].sum():,}")
print()
clusters[['cluster_label', 'n_total', 'success_rate', 'primary_failure_mode']].to_string(index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(clusters))
bars = ax.bar(x, clusters['success_rate'] * 100, color='#E03E3E', alpha=0.85)
ax.axhline(y=33, color='#F27A3A', linestyle='--', linewidth=1.5, label='Overall avg (33%)')

ax.set_xticks(x)
ax.set_xticklabels(clusters['cluster_label'], rotation=45, ha='right')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Success Rate by Skill Cluster (n=2,435)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 70)
ax.legend()

for bar, rate in zip(bars, clusters['success_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{rate*100:.0f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../figures/cluster-success-rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Failure Mode Breakdown

What causes skills to fail? The top two modes — missing prerequisites and ambiguous instructions — are both addressable by a type system.

In [ ]:
failure_modes = freq['failure_modes']

fig, ax = plt.subplots(figsize=(8, 8))
labels = [k.replace('_', ' ').title() for k in failure_modes.keys()]
sizes = list(failure_modes.values())
colors_pie = ['#E03E3E', '#F27A3A', '#F59E0B', '#3B82F6', '#9CA3AF', '#3D3D3D']

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.0f%%', startangle=90,
    colors=colors_pie
)
ax.set_title('Failure Mode Distribution (n=2,435)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/failure-modes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Key Insight: Type-Addressable Failures

The core argument for the Effector type system:

- **Missing prerequisites (28%)** → Typed `context` declarations catch this at compose-time
- **Ambiguous instructions (22%)** → Typed `input`/`output` force explicit interface contracts
- **Permission errors (11%)** → Declared `permissions` in effector.toml, auditable before execution

**61% of all failures are addressable by a type layer.** That's the case for Effector.

In [ ]:
type_addressable = 0.28 + 0.22 + 0.11
not_addressable = 1 - type_addressable

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([type_addressable, not_addressable],
       labels=['Type-addressable\n(prerequisites + instructions + permissions)',
               'Other\n(drift, timeout, misc)'],
       autopct='%1.0f%%', startangle=90,
       colors=['#E03E3E', '#3D3D3D'])
ax.set_title('Failures Addressable by Type System', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/type-addressable.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Type-addressable failures: {type_addressable:.0%}")
print(f"If Effector catches these, failure rate drops from 67% to ~26%")